# SynBADD qualitative comparison: NODER and ReMiND

This notebook discovers the newest available fold outputs on `F:/anirudhkaushik`, joins all methods by the exact ADNI Image Data ID and target visit, randomly selects five eligible subjects, and creates the three requested qualitative views. Re-run the selection cell with a different `RANDOM_SEED` to obtain another sample.

In [ ]:
# If needed, uncomment once:
# %pip install nibabel pandas matplotlib torch

from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = Path(r"F:\anirudhkaushik")
N_SUBJECTS = 5
RANDOM_SEED = 2026       # change this and rerun the selection cell
SELECTED_INDEX = 0       # 0..4, used by the SVF and slice-consistency cells
VECINT_STEPS = 7         # scaling-and-squaring steps for SVF integration

assert ROOT.exists(), f"Backup root is unavailable: {ROOT}"
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

In [ ]:
# Discovery and exact image-ID indexing. This cell is safe to rerun while files are still arriving.
IMAGE_ID_RE = re.compile(r"_I(\d+)_")
MAIN_MODEL_RE = re.compile(r"CrossAttentionTransformer\+DAFT_Past\+Future_Fold([1-5])$")

def image_id(path):
    m = IMAGE_ID_RE.search(path.name)
    return f"I{m.group(1)}" if m else None

def resolve_scratch_path(value):
    if pd.isna(value):
        return None
    raw = str(value).replace('\\', '/')
    prefix = '/scratch/anirudhkaushik/'
    rel = raw.split(prefix, 1)[1] if prefix in raw else raw.lstrip('/')
    for p in (ROOT / rel, ROOT / 'data' / rel, Path(str(value))):
        if p.exists():
            return p
    return ROOT / rel  # useful diagnostic path even if backup is incomplete

def choose_flat_result_dir(name):
    candidates = [ROOT / name, ROOT / 'data' / name]
    candidates = [p for p in candidates if p.is_dir()]
    if not candidates:
        return None
    # Prefer the most complete copy, then the newest one.
    return max(candidates, key=lambda p: (sum(1 for _ in p.glob('*.nii*')), p.stat().st_mtime))

def main_synbadd_candidates(fold):
    bases = [
        ROOT / f'SYNBADD_Qualitative_{fold}',
        ROOT / 'data' / f'SYNBADD_Qualitative_{fold}',
        ROOT / 'SYNBADD_outputs' / f'SYNBADD_Qualitative_{fold}',
    ]
    found = []
    for base in bases:
        if not base.is_dir():
            continue
        for p in [base, *[q for q in base.rglob('*') if q.is_dir()]]:
            if MAIN_MODEL_RE.search(p.name) and '_mask' not in p.name and 'experiment_' not in str(p):
                n_scans = sum(1 for q in p.glob('*_all_synthetic.nii.gz'))
                n_fields = sum(1 for q in p.glob('*_all_synthetic_field.nii.gz'))
                found.append((min(n_scans, n_fields), p.stat().st_mtime, p))
    return max(found, default=(0, 0, None))[2]

def index_outputs(folder, kind):
    out = {}
    if folder is None:
        return out
    for p in folder.glob('*.nii*'):
        iid = image_id(p)
        if iid is None:
            continue
        if kind == 'syn_scan' and p.name.endswith('_all_synthetic.nii.gz'):
            out[iid] = p
        elif kind == 'syn_field' and p.name.endswith('_all_synthetic_field.nii.gz'):
            out[iid] = p
        elif kind in {'noder', 'remind'}:
            out[iid] = p
    return out

meta_candidates = [ROOT / 'svf_metadata.csv', ROOT / 'data' / 'svf_metadata.csv']
META_PATH = max((p for p in meta_candidates if p.exists()), key=lambda p: p.stat().st_mtime)
metadata = pd.read_csv(META_PATH)

fold_dirs = {}
indexes = {}
for fold in range(1, 6):
    fold_dirs[fold] = {
        'SynBADD': main_synbadd_candidates(fold),
        'NODER': choose_flat_result_dir(f'NODER_result_{fold}'),
        'ReMiND': choose_flat_result_dir(f'remind_results_{fold}'),
    }
    indexes[fold] = {
        'syn': index_outputs(fold_dirs[fold]['SynBADD'], 'syn_scan'),
        'field': index_outputs(fold_dirs[fold]['SynBADD'], 'syn_field'),
        'noder': index_outputs(fold_dirs[fold]['NODER'], 'noder'),
        'remind': index_outputs(fold_dirs[fold]['ReMiND'], 'remind'),
    }

display(pd.DataFrame(fold_dirs).T.rename_axis('fold'))

In [ ]:
# Build eligible exact matches, choose one target visit per subject, then sample five subjects.
rows = []
for _, r in metadata.iterrows():
    iid = str(r['Image Data ID'])
    for fold in range(1, 6):
        if str(r.get(f'fold_{fold}', '')).lower() != 'test':
            continue
        ix = indexes[fold]
        if not all(iid in ix[k] for k in ('syn', 'field', 'noder', 'remind')):
            continue
        target = resolve_scratch_path(r['File_Path'])
        gt_svf = resolve_scratch_path(r['SVF_Path'])
        subject_rows = metadata[metadata['Subject'].astype(str) == str(r['Subject'])].copy()
        subject_rows['_visit_num'] = subject_rows['Visit'].astype(str).str.extract(r'(\d+)')[0].astype(float)
        baseline_row = subject_rows.sort_values('_visit_num').iloc[0]
        baseline = resolve_scratch_path(baseline_row['File_Path'])
        required = [target, baseline, ix['syn'][iid], ix['field'][iid], ix['noder'][iid], ix['remind'][iid]]
        if all(p is not None and p.exists() for p in required):
            rows.append({
                'subject': str(r['Subject']), 'group': r['Group'], 'visit': r['Visit'],
                'image_id': iid, 'fold': fold, 'baseline_visit': baseline_row['Visit'],
                'target': target, 'baseline': baseline, 'ground_truth_svf': gt_svf,
                'SynBADD': ix['syn'][iid], 'SynBADD_field': ix['field'][iid],
                'NODER': ix['noder'][iid], 'ReMiND': ix['remind'][iid],
            })

eligible = pd.DataFrame(rows)
if eligible.empty:
    raise RuntimeError('No complete exact matches yet. Re-run after more of the F: backup arrives.')
rng = np.random.default_rng(RANDOM_SEED)
subjects = rng.choice(eligible['subject'].unique(), size=min(N_SUBJECTS, eligible['subject'].nunique()), replace=False)
selected = (eligible[eligible.subject.isin(subjects)]
            .assign(_random=rng.random(sum(eligible.subject.isin(subjects))))
            .sort_values('_random').groupby('subject', as_index=False).first()
            .sort_values(['fold', 'subject']).reset_index(drop=True))
display(selected[['subject', 'group', 'visit', 'image_id', 'fold', 'baseline_visit']])
print(f'{len(eligible)} exact target matches across {eligible.subject.nunique()} subjects are currently available.')

In [ ]:
# Helpers and requested middle-axial comparison: Target | NODER | SynBADD | ReMiND
def load_volume(path, canonical=True):
    img = nib.load(str(path))
    if canonical:
        img = nib.as_closest_canonical(img)
    return np.asarray(img.dataobj, dtype=np.float32)

def robust01(vol):
    finite = vol[np.isfinite(vol)]
    lo, hi = np.percentile(finite, (0.5, 99.5))
    return np.clip((vol - lo) / max(hi - lo, 1e-8), 0, 1)

def axial(vol, z=None):
    z = vol.shape[2] // 2 if z is None else int(np.clip(z, 0, vol.shape[2] - 1))
    return np.rot90(vol[:, :, z])

methods = ['target', 'NODER', 'SynBADD', 'ReMiND']
fig, axes = plt.subplots(len(selected), 4, figsize=(10, 2.45 * len(selected)))
axes = np.atleast_2d(axes)
for row_i, (_, r) in enumerate(selected.iterrows()):
    for col_i, method in enumerate(methods):
        vol = robust01(load_volume(r[method]))
        axes[row_i, col_i].imshow(axial(vol), cmap='gray', vmin=0, vmax=1)
        axes[row_i, col_i].axis('off')
        if row_i == 0: axes[row_i, col_i].set_title('Target Scan' if method == 'target' else method)
    axes[row_i, 0].set_ylabel(f"{r.subject}\n{r.visit} | fold {r.fold}", rotation=0, labelpad=58, va='center')
fig.suptitle('Exact subject/timepoint comparison — middle axial slice', y=1.0)
fig.tight_layout()
plt.show()

In [ ]:
# Difference maps: prediction minus target after robust [0,1] intensity normalization.
fig, axes = plt.subplots(len(selected), 3, figsize=(8.2, 2.45 * len(selected)))
axes = np.atleast_2d(axes)
for row_i, (_, r) in enumerate(selected.iterrows()):
    target = robust01(load_volume(r.target))
    differences = []
    for method in ['NODER', 'SynBADD', 'ReMiND']:
        pred = robust01(load_volume(r[method]))
        if pred.shape != target.shape:
            raise ValueError(f"Shape mismatch for {r.image_id}: target {target.shape}, {method} {pred.shape}")
        differences.append(pred - target)
    scale = max(np.percentile(np.abs(d), 99) for d in differences)
    for col_i, (method, diff) in enumerate(zip(['NODER', 'SynBADD', 'ReMiND'], differences)):
        im = axes[row_i, col_i].imshow(axial(diff), cmap='seismic', vmin=-scale, vmax=scale)
        axes[row_i, col_i].axis('off')
        if row_i == 0: axes[row_i, col_i].set_title(f'{method} − Target')
    axes[row_i, 0].set_ylabel(f"{r.subject}\n{r.visit}", rotation=0, labelpad=52, va='center')
    fig.colorbar(im, ax=axes[row_i, :], fraction=0.018, pad=0.01)
fig.suptitle('Signed difference maps — per-subject common color scale', y=1.0)
fig.tight_layout()
plt.show()

In [ ]:
# SynBADD deformation view: baseline, target, warped scan, target det(J), generated det(J).
# Both SVFs are integrated by scaling-and-squaring before det(J) is calculated.
from synbadd.models.spatial_ops import VecInt

def field_channels_first(value):
    if isinstance(value, dict):
        for key in ('svf', 'field', 'flow', 'velocity', 'disp'):
            if key in value:
                value = value[key]; break
        else:
            value = next(v for v in value.values() if hasattr(v, 'shape'))
    a = value.detach().cpu().numpy() if torch.is_tensor(value) else np.asarray(value)
    a = np.squeeze(a)
    if a.ndim != 4:
        raise ValueError(f'Expected a 3-D vector field, got {a.shape}')
    if a.shape[0] == 3:
        return a.astype(np.float32)
    if a.shape[-1] == 3:
        return np.moveaxis(a, -1, 0).astype(np.float32)
    raise ValueError(f'Cannot locate vector components in {a.shape}')

def load_svf(path):
    if path is None or not Path(path).exists():
        raise FileNotFoundError(f'SVF is not backed up yet: {path}')
    if str(path).endswith('.pt'):
        try:
            value = torch.load(path, map_location='cpu', weights_only=True)
        except TypeError:
            value = torch.load(path, map_location='cpu')
    else:
        value = np.asarray(nib.load(str(path)).dataobj)
    return field_channels_first(value)

def integrate_svf(svf, steps=VECINT_STEPS):
    x = torch.from_numpy(svf[None]).float()
    with torch.no_grad():
        return VecInt(tuple(svf.shape[1:]), steps)(x)[0].cpu().numpy()

def jacobian_determinant(displacement):
    # displacement component order follows the model's (D,H,W) convention.
    g = [[np.gradient(displacement[i], axis=j) for j in range(3)] for i in range(3)]
    j00, j01, j02 = 1 + g[0][0], g[0][1], g[0][2]
    j10, j11, j12 = g[1][0], 1 + g[1][1], g[1][2]
    j20, j21, j22 = g[2][0], g[2][1], 1 + g[2][2]
    return j00*(j11*j22-j12*j21) - j01*(j10*j22-j12*j20) + j02*(j10*j21-j11*j20)

r = selected.iloc[SELECTED_INDEX]
gt_det = jacobian_determinant(integrate_svf(load_svf(r.ground_truth_svf)))
pred_det = jacobian_determinant(integrate_svf(load_svf(r.SynBADD_field)))
panels = [
    ('Baseline Scan', axial(robust01(load_volume(r.baseline)))),
    ('Target Scan', axial(robust01(load_volume(r.target)))),
    ('Warped Scan', axial(robust01(load_volume(r.SynBADD)))),
    ('Target det(J)', axial(gt_det)),
    ('Synthesized det(J)', axial(pred_det)),
]
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
det_lo = min(np.percentile(gt_det, 1), np.percentile(pred_det, 1))
det_hi = max(np.percentile(gt_det, 99), np.percentile(pred_det, 99))
for i, (title, image) in enumerate(panels):
    if i < 3: axes[i].imshow(image, cmap='gray', vmin=0, vmax=1)
    else: axes[i].imshow(image, cmap='turbo', vmin=det_lo, vmax=det_hi)
    axes[i].set_title(title); axes[i].axis('off')
fig.suptitle(f'{r.subject} — target {r.visit} (baseline {r.baseline_visit})')
fig.tight_layout()
plt.show()

In [ ]:
# SOTA slice-level consistency around the middle axial slice for one selected case.
r = selected.iloc[SELECTED_INDEX]
volumes = {m: robust01(load_volume(r[m])) for m in ['target', 'NODER', 'SynBADD', 'ReMiND']}
center = volumes['target'].shape[2] // 2
slice_ids = list(range(center - 2, center + 3))
fig, axes = plt.subplots(5, 4, figsize=(8.5, 10.5))
for row_i, z in enumerate(slice_ids):
    for col_i, method in enumerate(['target', 'NODER', 'SynBADD', 'ReMiND']):
        axes[row_i, col_i].imshow(axial(volumes[method], z), cmap='gray', vmin=0, vmax=1)
        axes[row_i, col_i].axis('off')
        if row_i == 0: axes[row_i, col_i].set_title('Target Scan' if method == 'target' else method)
    axes[row_i, 0].set_ylabel(f'Slice {z}', rotation=0, labelpad=36, va='center')
fig.suptitle(f'Slice-level consistency: {r.subject}, target {r.visit}', y=1.0)
fig.tight_layout()
plt.show()